In [1]:
# =====================================================================
# SEL 1: IMPORT LIBRARY DAN LOAD DATA READY-MODELING
# =====================================================================
import os
import pandas as pd
import mlflow
from pycaret.regression import setup, compare_models, pull, set_config, tune_model, evaluate_model

# Menggunakan relative path yang aman karena notebook berada di folder 'notebooks'
file_path = os.path.abspath(os.path.join("..", "data", "processed", "final_dataset.csv"))

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ Data berhasil dimuat. Total siap modeling: {len(df)} baris.")
else:
    print(f"❌ File tidak ditemukan di: {file_path}")
    print("Pastikan kamu sudah menjalankan tahap preprocessing.")

display(df.head())

✅ Data berhasil dimuat. Total siap modeling: 13026 baris.


,time,location,wave_height,wind_speed_10m,precipitation,visibility,ocean_current_velocity,sea_surface_temperature,wave_height_lag1,wave_height_lag2,...,precipitation_lag3,visibility_lag1,visibility_lag2,visibility_lag3,ocean_current_velocity_lag1,ocean_current_velocity_lag2,ocean_current_velocity_lag3,sea_surface_temperature_lag1,sea_surface_temperature_lag2,sea_surface_temperature_lag3
0,2026-05-08 03:00:00,Bangkalan,0.2,7.1,0.0,40400.0,1.3,31.0,0.2,0.2,...,0.0,37700.0,37700.0,37700.0,0.9,0.4,0.4,31.1,31.1,31.1
1,2026-05-08 04:00:00,Bangkalan,0.2,5.4,0.0,36980.0,1.5,31.0,0.2,0.2,...,0.0,40400.0,37700.0,37700.0,1.3,0.9,0.4,31.0,31.1,31.1
2,2026-05-08 05:00:00,Bangkalan,0.2,8.5,0.0,34320.0,1.5,31.0,0.2,0.2,...,0.0,36980.0,40400.0,37700.0,1.5,1.3,0.9,31.0,31.0,31.1
3,2026-05-08 06:00:00,Bangkalan,0.2,7.5,0.0,34320.0,1.1,31.0,0.2,0.2,...,0.0,34320.0,36980.0,40400.0,1.5,1.5,1.3,31.0,31.0,31.0
4,2026-05-08 07:00:00,Bangkalan,0.2,4.0,0.0,37660.0,0.7,31.0,0.2,0.2,...,0.0,34320.0,34320.0,36980.0,1.1,1.5,1.5,31.0,31.0,31.0


In [2]:
# =====================================================================
# SEL 2: MODELING LOOP & COMPARISON (TOP 3 PER TARGET VARIABLE)
# =====================================================================
# Menyiapkan direktori MLflow
mlflow_tracking_uri = f"file:///{os.path.abspath(os.path.join('..', 'mlruns'))}"
mlflow.set_tracking_uri(mlflow_tracking_uri)
print(f"📁 MLflow Tracking diaktifkan di: {mlflow_tracking_uri}")

# Menghapus fitur identitas agar tidak memicu data leakage
base_df = df.drop(columns=["time", "location", "id_location"], errors='ignore')

all_results = {}
features = [
    "wave_height", "wind_speed_10m", "ocean_current_velocity", 
    "sea_surface_temperature", "precipitation", "visibility"
]

for target in features:
    print(f"\n{'='*20} PROSES SELECTION TARGET: {target.upper()} {'='*20}")
    
    # Pencegahan Data Leakage (Buang target lain)
    other_targets = [col for col in features if col != target]
    target_df = base_df.drop(columns=other_targets, errors='ignore')
    
    # Setup PyCaret
    s = setup(
        data=target_df,
        target=target,
        session_id=123,
        fold=5,
        train_size=0.8,
        verbose=False,
        log_experiment=True, 
        experiment_name=f"Baseline_{target}"
    )
    
    # Mengambil 3 model terbaik berdasarkan RMSE
    best_models = compare_models(n_select=3, sort="RMSE")
    
    # Ekstraksi leaderboard
    results = pull()
    all_results[target] = {"results": results}
    
    print(f"Top 3 Model Terbaik untuk {target}:")
    display(results.head(3)[["Model", "MAE", "RMSE", "R2"]])

# --- KOMPILASI RINGKASAN TOP 3 MODEL ---
compiled_rows = []
for target, data in all_results.items():
    top_3 = data["results"].head(3).copy()
    top_3.insert(0, "Target Variable", target)
    compiled_rows.append(top_3)

pycaret_summary_df = pd.concat(compiled_rows, ignore_index=True)

# Format visualisasi tabel akhir
styled_summary = pycaret_summary_df.style.background_gradient(
    subset=["R2"], cmap="YlGn"
).background_gradient(
    subset=["MAE", "RMSE"], cmap="Reds"
).format({
    "R2": "{:.4f}", "MAE": "{:.4f}", "RMSE": "{:.4f}", "MSE": "{:.4f}", "MAPE": "{:.4f}"
})

print("\n" + "="*50)
print("--- TABEL PERBANDINGAN TOP 3 MODEL PYCARET UNTUK SETIAP TARGET ---")
print("="*50)
display(styled_summary)

# Opsional: Simpan tabel laporan ke CSV untuk presentasi
reports_dir = os.path.abspath(os.path.join("..", "reports"))
os.makedirs(reports_dir, exist_ok=True)
pycaret_summary_df.to_csv(os.path.join(reports_dir, "pycaret_top3_summary.csv"), index=False)

📁 MLflow Tracking diaktifkan di: file:///d:\LENTERA_LAUT\mlruns

==================== PROSES SELECTION TARGET: WAVE_HEIGHT ====================


2026/05/28 09:39:57 INFO mlflow.tracking.fluent: Experiment with name 'Baseline_wave_height' does not exist. Creating a new experiment.


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lr,Linear Regression,0.0095,0.0002,0.0136,0.9994,0.0083,0.0261,0.8300
lightgbm,Light Gradient Boosting Machine,0.0099,0.0002,0.0145,0.9994,0.0085,0.0259,0.2380
xgboost,Extreme Gradient Boosting,0.0103,0.0002,0.0147,0.9994,0.0086,0.0274,0.3240
rf,Random Forest Regressor,0.0102,0.0002,0.0148,0.9993,0.0087,0.0264,0.9900
et,Extra Trees Regressor,0.0100,0.0002,0.0148,0.9993,0.0087,0.0258,0.4200
ridge,Ridge Regression,0.0098,0.0002,0.0148,0.9993,0.0089,0.0248,0.4720
gbr,Gradient Boosting Regressor,0.0108,0.0003,0.0163,0.9992,0.0095,0.0273,0.4820
br,Bayesian Ridge,0.0123,0.0003,0.0183,0.9990,0.0111,0.0346,0.0140
dt,Decision Tree Regressor,0.0125,0.0004,0.0200,0.9988,0.0117,0.0310,0.0360
ada,AdaBoost Regressor,0.0280,0.0013,0.0366,0.9960,0.0279,0.1726,0.2520


2026/05/28 09:40:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 09:40:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/28 09:40:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 09:40:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Top 3 Model Terbaik untuk wave_height:


...

2026/05/28 09:44:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 09:44:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/28 09:44:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 09:44:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Top 3 Model Terbaik untuk visibility:


,Model,MAE,RMSE,R2
gbr,Gradient Boosting Regressor,4013.1797,6154.1778,0.7609
lightgbm,Light Gradient Boosting Machine,4020.8074,6175.4955,0.7594
ridge,Ridge Regression,3987.8430,6227.0229,0.7553



--- TABEL PERBANDINGAN TOP 3 MODEL PYCARET UNTUK SETIAP TARGET ---


,Target Variable,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
0,wave_height,Linear Regression,0.0095,0.0002,0.0136,0.9994,0.008300,0.0261,0.830000
1,wave_height,Light Gradient Boosting Machine,0.0099,0.0002,0.0145,0.9994,0.008500,0.0259,0.238000
2,wave_height,Extreme Gradient Boosting,0.0103,0.0002,0.0147,0.9994,0.008600,0.0274,0.324000
3,wind_speed_10m,Light Gradient Boosting Machine,1.5012,4.0274,2.0063,0.8324,0.302500,0.3652,0.178000
4,wind_speed_10m,Extra Trees Regressor,1.5019,4.0725,2.0174,0.8306,0.302800,0.3641,1.124000
5,wind_speed_10m,Random Forest Regressor,1.5062,4.0976,2.0236,0.8295,0.303900,0.3672,1.728000
6,ocean_current_velocity,Extra Trees Regressor,0.0918,0.0237,0.1538,0.9848,0.080200,0.1684,1.388000
7,ocean_current_velocity,Light Gradient Boosting Machine,0.1002,0.0286,0.1692,0.9816,0.085000,0.1689,0.482000
8,ocean_current_velocity,Extreme Gradient Boosting,0.1026,0.0296,0.1721,0.9810,0.085900,0.1798,0.154000
9,sea_surface_temperature,Extra Trees Regressor,0.0366,0.0027,0.0519,0.9985,0.001700,0.0012,0.654000
